# 03 — MLflow Sessions: Multi-Turn Chat Tracking

**UI tab:** Traces → Sessions

A **session** groups multiple traces that belong to the same conversation.
Each turn creates one trace. Calling `mlflow.update_current_trace(session_id=...)`
inside a `@mlflow.trace`-decorated function links every turn to one session.

```
Session abc-123
├── Trace: turn 1  →  ask_gemini("What is MLflow?")
├── Trace: turn 2  →  ask_gemini("How does tracing work?")
└── Trace: turn 3  →  ask_gemini("What are sessions?")
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai --quiet

In [ ]:
import os, uuid
from google import genai
from google.genai import types
import mlflow

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.gemini.autolog()          # auto-trace all Gemini calls as child spans
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("03-MLflow-Sessions")

print("MLflow", mlflow.__version__, "ready")

## Session 1 — 3 turns on MLflow topics

In [ ]:
# Each session gets a unique ID — all traces tagged with the same ID
# appear grouped under Traces → Sessions in the MLflow UI
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}")

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are an MLflow expert. Answer in 2 sentences max."
    )
)

@mlflow.trace
def chat_turn(user_msg: str) -> str:
    # Link this trace to session_id — MLflow groups traces by session in the UI
    mlflow.update_current_trace(session_id=session_id)
    response = chat.send_message(user_msg)
    return (response.text or '').strip()

turns = [
    "What is MLflow?",
    "What is the Traces tab in MLflow?",
    "What is an MLflow session?"
]

for i, msg in enumerate(turns, 1):
    answer = chat_turn(msg)
    print(f"[Turn {i}] User: {msg}")
    print(f"         Bot:  {answer}\n")

print("\nOpen MLflow UI → Traces → Sessions → you should see session_id")
print("with 3 traces nested under it, one per turn.")

## Session 2 — 2 turns on Python topics

In [ ]:
session_id_2 = str(uuid.uuid4())
print(f"Session 2 ID: {session_id_2}")

chat2 = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a Python tutor. Answer in 2 sentences max."
    )
)

@mlflow.trace
def chat_turn_2(user_msg: str) -> str:
    mlflow.update_current_trace(session_id=session_id_2)
    response = chat2.send_message(user_msg)
    return (response.text or '').strip()

turns2 = [
    "What is a Python list?",
    "How is it different from a tuple?"
]

for i, msg in enumerate(turns2, 1):
    answer = chat_turn_2(msg)
    print(f"[Turn {i}] User: {msg}")
    print(f"         Bot:  {answer}\n")

print(f"\nSessions tab now shows 2 sessions:")
print(f"  Session 1 ({session_id[:8]}...): 3 MLflow turns")
print(f"  Session 2 ({session_id_2[:8]}...): 2 Python turns")

## MLflow UI — What to explore
```
Traces tab → Sessions view
├── session-<id1>   →  3 turns on MLflow topics
└── session-<id2>   →  2 turns on Python

Click any session → see the full conversation thread
Click any turn   → see full Gemini request / response span
```
**Next →** `04_mlflow_judges_evaluation.ipynb`